In [33]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Generate sample data (continuous and dummy variables)
def generate_sample_data(num_samples=1000, num_features=5):
    np.random.seed(42)
    data = np.random.normal(loc=0, scale=1, size=(num_samples, num_features))
    # Normalize non-dummy data to have values between 0 and 1
    for i in range(num_features - 2):
        min_val = data[:, i].min()
        max_val = data[:, i].max()
        data[:, i] = (data[:, i] - min_val) / (max_val - min_val)
    # Make the last two features dummy (binary 0/1)
    data[:, -2:] = np.random.randint(0, 2, size=(num_samples, 2))
    return torch.tensor(data, dtype=torch.float32)

# Function to detect continuous and dummy dimensions
def detect_dimensions(data):
    continuous_columns = []
    dummy_columns = []

    for col in range(data.shape[1]):
        unique_values = np.unique(data[:, col])
        if set(unique_values).issubset({0, 1}):  # Dummy variable condition
            dummy_columns.append(col)
        else:
            continuous_columns.append(col)

    return continuous_columns, dummy_columns

# Generator network
class Generator(nn.Module):
    def __init__(self, latent_dim, continuous_dim, categorical_dim):
        super(Generator, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.continuous_layer = nn.Linear(32, continuous_dim)
        self.categorical_layer = nn.Linear(32, categorical_dim)

    def forward(self, z):
        x = self.fc(z)
        continuous = self.continuous_layer(x)
        categorical = torch.sigmoid(self.categorical_layer(x))  # Sigmoid for dummy variables
        return torch.cat([continuous, categorical], dim=1)

# Discriminator network
class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.fc(x)

# Training function
def train_gan(generator, discriminator, data, latent_dim, epochs=1000, batch_size=32, lr=0.0002):
    optimizer_G = optim.Adam(generator.parameters(), lr=lr)
    optimizer_D = optim.Adam(discriminator.parameters(), lr=lr)
    criterion = nn.BCELoss()

    for epoch in range(epochs):
        # Sample real data
        idx = np.random.randint(0, data.size(0), batch_size)
        real_data = data[idx]

        # Train Discriminator
        real_labels = torch.ones((batch_size, 1))
        fake_labels = torch.zeros((batch_size, 1))

        z = torch.randn(batch_size, latent_dim)
        fake_data = generator(z).detach()

        real_preds = discriminator(real_data)
        fake_preds = discriminator(fake_data)

        d_loss = criterion(real_preds, real_labels) + criterion(fake_preds, fake_labels)
        optimizer_D.zero_grad()
        d_loss.backward()
        optimizer_D.step()

        # Train Generator
        z = torch.randn(batch_size, latent_dim)
        fake_data = generator(z)
        fake_preds = discriminator(fake_data)

        g_loss = criterion(fake_preds, real_labels)
        optimizer_G.zero_grad()
        g_loss.backward()
        optimizer_G.step()

        # Print losses
        if (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch + 1}/{epochs}, D Loss: {d_loss.item():.4f}, G Loss: {g_loss.item():.4f}")

        # Assess training quality and balance
        if (epoch + 1) % 200 == 0:
            if d_loss.item() < 1.0 and g_loss.item() > 0.9:
                print("Model is well-balanced. Generator is producing realistic data.")
            elif d_loss.item() > 1.5:
                print("Discriminator is dominating. Generator needs improvement.")
            elif g_loss.item() < 0.7:
                print("Generator is struggling. Increase training epochs.")

# Sampling function
def generate_samples(generator, num_samples, latent_dim):
    z = torch.randn(num_samples, latent_dim)
    samples = generator(z).detach().numpy()
    # Post-process dummy variables
    samples[:, -2:] = (np.random.rand(num_samples, 2) > 0.5).astype(int)  # Randomize dummy outputs to introduce variability
    return samples

# Parameters
latent_dim = 5

data = generate_sample_data()
data_np = data.numpy()
continuous_columns, dummy_columns = detect_dimensions(data_np)

continuous_dim = len(continuous_columns)
categorical_dim = len(dummy_columns)

generator = Generator(latent_dim, continuous_dim, categorical_dim)
discriminator = Discriminator(continuous_dim + categorical_dim)

# Train GAN
train_gan(generator, discriminator, data, latent_dim, epochs=1000, batch_size=32)

# Generate new samples
samples = generate_samples(generator, num_samples=10, latent_dim=latent_dim)
print("Generated Samples:")
print(samples)


Epoch 100/1000, D Loss: 1.3209, G Loss: 0.8110
Epoch 200/1000, D Loss: 1.3720, G Loss: 0.7031
Epoch 300/1000, D Loss: 1.3005, G Loss: 0.7533
Epoch 400/1000, D Loss: 1.2621, G Loss: 0.8298
Epoch 500/1000, D Loss: 1.2364, G Loss: 0.8314
Epoch 600/1000, D Loss: 1.1256, G Loss: 0.9063
Epoch 700/1000, D Loss: 0.8894, G Loss: 1.0801
Epoch 800/1000, D Loss: 0.8754, G Loss: 1.0730
Model is well-balanced. Generator is producing realistic data.
Epoch 900/1000, D Loss: 0.8837, G Loss: 1.2578
Epoch 1000/1000, D Loss: 0.5905, G Loss: 1.4043
Model is well-balanced. Generator is producing realistic data.
Generated Samples:
[[ 0.4755356  -0.34723312  0.75389457  0.          1.        ]
 [ 0.9104052   0.21666673  1.0678649   0.          0.        ]
 [ 0.8745061   0.26720396  1.7975405   1.          0.        ]
 [ 0.6040045  -0.01697864  0.8812464   1.          0.        ]
 [ 0.44175872 -0.11304168  0.42344055  0.          0.        ]
 [ 0.49582228 -0.19242245  0.57465047  0.          1.        ]
 [ 0.7

In [29]:
data

tensor([[0.5167, 0.4532, 0.6083, 1.0000, 0.0000],
        [0.4138, 0.7309, 0.6270, 1.0000, 1.0000],
        [0.3814, 0.4002, 0.5448, 1.0000, 0.0000],
        ...,
        [0.6773, 0.4398, 0.5178, 0.0000, 0.0000],
        [0.1509, 0.5860, 0.4890, 1.0000, 0.0000],
        [0.4398, 0.5906, 0.9939, 1.0000, 1.0000]])